# Failure Mode 2: Goal Achievement

> **New to agent evaluation?** Read the [project README](../../README.md) first — it explains the key concepts (traces, failure modes, scorers, expectations) that these notebooks build on.

The agent finishes but doesn't actually satisfy what the user asked for — partial answers, wrong answers, or failure to engage. This is the most fundamental evaluation: did the agent do what it was supposed to do?

We evaluate this using four scorers that detect the failure from different angles:

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| `Correctness` | MLflow native | Yes | Are specific expected facts in the response text? |
| `AgentGoalAccuracyWithReference` | RAGAS via MLflow | Yes | Does the workflow outcome match the reference? (sees full conversation) |
| `TaskCompletion` | DeepEval via MLflow | No | Did the agent accomplish the task? (workflow-focused) |
| `AgentGoalAccuracyWithoutReference` | RAGAS via MLflow | No | Did the agent achieve the goal AND communicate it? |

For a detailed explanation of this failure mode, how each scorer works internally, and the key insight that **retrieving the data ≠ communicating it to the user**, see [goal_achievement.md](goal_achievement.md).

### Prerequisites and setup

Start a local MLflow server before running this notebook:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import SpanType
from tools import WEATHER_AGENT_TOOLS
from utils import print_eval_results

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("agentic-evaluation")
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name("agentic-evaluation")

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'goal_achievement'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Create traces

We create synthetic traces for a weather agent answering "What's the weather and humidity in Paris?" Both traces call the same tool and get the same data — the difference is in how the agent formulates its response:

- **Partial answer (fail):** Agent reports only the temperature, omitting humidity and condition
- **Complete answer (pass):** Agent includes all requested details

In [ ]:
@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def goal_achievement_partial_answer(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, WEATHER_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "goal_achievement", "expected_result": "fail"}
    )

    with mlflow.start_span(name="get_weather", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "Paris"})
        span.set_outputs({
            "temperature_celsius": 22,
            "condition": "partly cloudy",
            "humidity": 65,
        })

    return "It's 22°C in Paris."


@mlflow.trace(name="weather_agent", span_type=SpanType.AGENT)
def goal_achievement_complete_answer(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, WEATHER_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "goal_achievement", "expected_result": "pass"}
    )

    with mlflow.start_span(name="get_weather", span_type=SpanType.TOOL) as span:
        span.set_inputs({"location": "Paris"})
        span.set_outputs({
            "temperature_celsius": 22,
            "condition": "partly cloudy",
            "humidity": 65,
        })

    return "It's 22°C and partly cloudy in Paris, with 65% humidity."


user_msg = [{"role": "user", "content": "What's the weather and humidity in Paris?"}]
goal_achievement_partial_answer(user_msg)
goal_achievement_complete_answer(user_msg)
mlflow.flush_trace_async_logging()
print("Created 2 traces (1 fail, 1 pass)")

### Load traces

We fetch the Goal Achievement traces — one where the agent gives a partial answer (omits humidity), and one where it gives a complete answer.

In [ ]:
goal_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'goal_achievement'",
    return_type="list",
)

print(f"Traces found: {len(goal_traces)}")
for t in goal_traces:
    tags = t.info.tags or {}
    print(
        f"  [{tags.get('expected_result', '?')}] Input: {str(t.info.request_preview)[:80]}"
    )
    print(f"    Output: {str(t.info.response_preview)[:80]}")
    print()

### Approach 1: Correctness (MLflow native) — with expectations

`Correctness` checks whether specific expected facts are present in the agent's response. It only looks at the input and output — not tool spans. Requires `expected_facts`, which we log on the traces using `mlflow.log_expectation()`.

In [ ]:
# Log expectations and evaluate in the same cell to keep format consistent.
from mlflow.genai.scorers import Correctness

expected_facts = [
    "The temperature is 22°C",
    "The condition is partly cloudy",
    "The humidity is 65%",
]

for t in goal_traces:
    mlflow.log_expectation(
        trace_id=t.info.trace_id,
        name="expected_facts",
        value=expected_facts,
    )

# Re-fetch traces with expectations attached
goal_traces_with_facts = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'goal_achievement'",
    return_type="list",
)

with mlflow.start_run(run_name="goal-achievement-correctness") as run:
    results_correctness = mlflow.genai.evaluate(
        data=goal_traces_with_facts,
        scorers=[Correctness(model="openai:/gpt-4o")],
    )

print_eval_results(results_correctness, "correctness", EXPERIMENT.experiment_id)

### Approach 2: AgentGoalAccuracyWithReference (RAGAS via MLflow) — with expectations

`AgentGoalAccuracyWithReference` checks if the workflow outcome matches a reference answer you provide. Unlike `Correctness` which only sees input + response text, this scorer reads the **full conversation** (messages + tool calls) — making it more appropriate for agents that produce results through actions, not just text.

We log the reference as an `expected_output` expectation on the traces.

In [ ]:
# Log expectations and evaluate in the same cell to keep format consistent.
from mlflow.genai.scorers.ragas import AgentGoalAccuracyWithReference

for t in goal_traces:
    mlflow.log_expectation(
        trace_id=t.info.trace_id,
        name="expected_output",
        value="The weather in Paris is 22 degrees celsius, partly cloudy, with 65 percent humidity.",
    )

# Re-fetch traces with expectations attached
goal_traces_with_ref = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'goal_achievement'",
    return_type="list",
)

with mlflow.start_run(run_name="goal-achievement-with-reference") as run:
    results_with_ref = mlflow.genai.evaluate(
        data=goal_traces_with_ref,
        scorers=[AgentGoalAccuracyWithReference(model="openai:/gpt-4o")],
    )

print_eval_results(
    results_with_ref, "AgentGoalAccuracyWithReference", EXPERIMENT.experiment_id
)

### Approach 3: TaskCompletion (DeepEval via MLflow) — without expectations

`TaskCompletion` assesses whether the agent accomplished the task based on input, response, and tool calls. No expectations needed. Note: this scorer may give a passing score to partial answers if the tool returned the correct data — see [goal_achievement.md](goal_achievement.md) for why.

In [ ]:
from mlflow.genai.scorers.deepeval import TaskCompletion

with mlflow.start_run(run_name="goal-achievement-task-completion") as run:
    results_tc = mlflow.genai.evaluate(
        data=goal_traces,
        scorers=[TaskCompletion(model="openai:/gpt-4.1-mini")],
    )

print_eval_results(results_tc, "TaskCompletion", EXPERIMENT.experiment_id)

### Approach 4: AgentGoalAccuracyWithoutReference (RAGAS via MLflow) — without expectations

`AgentGoalAccuracyWithoutReference` checks both the workflow and the final response — it infers the user's goal and whether the agent communicated the result. No expectations needed. Catches partial answers that TaskCompletion misses.

In [ ]:
from mlflow.genai.scorers.ragas import AgentGoalAccuracyWithoutReference

with mlflow.start_run(run_name="goal-achievement-ragas") as run:
    results_ragas = mlflow.genai.evaluate(
        data=goal_traces,
        scorers=[AgentGoalAccuracyWithoutReference(model="openai:/gpt-4o")],
    )

print_eval_results(
    results_ragas, "AgentGoalAccuracyWithoutReference", EXPERIMENT.experiment_id
)

### Interpreting the results

- **Correctness:** Failing trace → `no` (missing facts), passing trace → `yes`. Checks response text only.
- **AgentGoalAccuracyWithReference:** Failing trace → `0.0`, passing trace → `1.0`. Checks full conversation against reference — catches partial answers even when the workflow was correct.
- **TaskCompletion:** Both traces may score `yes` / 1.0 — even for the partial answer, because the tool returned the data. Notice the rationale for the partial answer: it claims the system "provided the current temperature, weather description, and humidity" — but the agent only said "It's 22°C in Paris." The rationale is describing what the *tool returned*, not what the *user received*. **Retrieving the data ≠ communicating it to the user.**
- **AgentGoalAccuracyWithoutReference:** Failing trace → `0.0`, passing trace → `1.0` — catches the partial answer without needing expectations.

**Correctness vs AgentGoalAccuracyWithReference:** Both need expectations and both catch partial answers. The difference is what they see — Correctness checks if facts appear in the **response text**, while WithReference reads the **full conversation** (messages + tool calls) and compares the outcome against your reference. For agents that produce results through actions, WithReference is more appropriate.

**Note on models:** Approaches 1, 2, and 4 use `gpt-4o`. Approach 3 (TaskCompletion, DeepEval) uses `gpt-4.1-mini` because DeepEval 3.9.9 has a JSON parsing issue with `gpt-4o` responses. You can change the `model=` parameter to use any provider MLflow supports.

For a full comparison of what each scorer sees and when to use which, see [goal_achievement.md](goal_achievement.md).